In [ ]:
# Standard library imports
from datetime import datetime
from time import sleep

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from xgboost import XGBRegressor


# Local imports
from cpu_temp_bundled import HardwareMonitor

# Background color white
px.defaults.template = "plotly_white"

In [ ]:
class CoreTempRegressor:
    def __init__(self, use_time_features=False, lag_steps = [1, 2, 3, 5, 10], rolling_windows = [3, 5, 10, 20]):
        self.scaler = StandardScaler()
        self.data = None
        self.predict_data = None
        
        # Store for plot_predictions
        self.x_test = None
        self.y_test = None
        self.y_pred = None
        
        # Store feature engineering parameters
        self.lag_steps = lag_steps
        self.rolling_windows = rolling_windows
        self.use_time_features = use_time_features

    def extract_CPU_data(self, iterations=100, csv=False):
        if csv:
            self.data = pd.read_csv('data.csv')
            # Convert timestamp back to datetime if loading from CSV
            if 'timestamp' in self.data.columns:
                self.data['timestamp'] = pd.to_datetime(self.data['timestamp'])
        else:
            data_list = []
            with HardwareMonitor() as monitor:
                for t in range(iterations):
                    row = monitor.get_essential_fast()
                    row['timestamp'] = datetime.now()
                    row['time'] = t
                    data_list.append(row)
                    sleep(1)

            self.data = pd.DataFrame(data_list)
        return self.data

    def create_time_features_on_df(self, df, lag_steps, rolling_windows):
        exclude_cols = ['cpu_temp', 'time', 'fan_rpm', 'timestamp']
        feature_cols = [col for col in df.columns if col not in exclude_cols and df[col].dtype in ['float64', 'int64', 'float32', 'int32']]

        new_features = {}

        # Create lag and rolling features ONLY for independent variables
        for col in feature_cols:
            # Lag features
            for lag in lag_steps:
                new_features[f'{col}_lag_{lag}'] = df[col].shift(lag)

            # Rolling statistics
            for window in rolling_windows:
                new_features[f'{col}_rolling_mean_{window}'] = df[col].rolling(window=window).mean()
                new_features[f'{col}_rolling_std_{window}'] = df[col].rolling(window=window).std()

            # Rate of change
            new_features[f'{col}_diff_1'] = df[col].diff(1)

        if new_features:
            new_cols_df = pd.DataFrame(new_features, index=df.index)
            df = pd.concat([df, new_cols_df], axis=1)

        return df

    def plot_CPU_data(self):
        px.line(self.data, x='timestamp', y=['cpu_temp','cpu_load','cpu_power','cpu_clock','cpu_volt','gpu_temp','gpu_load','gpu_power','mb_temp','ram_load','fan_rpm'], title='CPU Temperature').show()

    def fit_predict(self, model='xgb', train_size=0.8):
        model_dict = {
            'xgb': XGBRegressor(
                n_estimators=1200,
                max_depth=5,
                learning_rate=0.005,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                random_state=42
            ),

            'linear': Ridge(
                solver='auto', 
                random_state=42
            ),

            'lightgbm': LGBMRegressor(
                n_estimators=1200,
                max_depth=5,
                learning_rate=0.005,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_samples=20,
                random_state=42,
                verbose=-1
            )
        }
        self.model = model_dict.get(model)
        
        # Split the raw data FIRST (before feature engineering)
        split_idx = int(len(self.data) * train_size)
        train_data = self.data[:split_idx].copy()
        test_data = self.data[split_idx:].copy()
        
        # Apply time features SEPARATELY to train and test
        if self.use_time_features:
            train_data = self.create_time_features_on_df(train_data, self.lag_steps, self.rolling_windows)
            test_data = self.create_time_features_on_df(test_data, self.lag_steps, self.rolling_windows)
            
            # Drop NaN rows created by lag/rolling (only affects beginning of each split)
            train_data = train_data.dropna()
            test_data = test_data.dropna()
        
        # Prepare X and y
        x_train = train_data.drop(['cpu_temp', 'time', 'fan_rpm', 'timestamp'], axis=1)
        y_train = train_data['cpu_temp']
        
        self.x_test = test_data.drop(['cpu_temp', 'time', 'fan_rpm', 'timestamp'], axis=1)
        self.y_test = test_data['cpu_temp']
        
        # Fit scaler ONLY on training data
        x_train_scaled = self.scaler.fit_transform(x_train)
        x_test_scaled = self.scaler.transform(self.x_test)
        
        # Train model
        self.model.fit(x_train_scaled, y_train)
        
        # Make predictions
        self.y_pred = self.model.predict(x_test_scaled)
        
        # Store test_data for plotting
        self.test_data = test_data
        
    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        self.predict_data = self.model.predict(X_scaled)
        return self.predict_data

    def plot_predictions(self):
        # Use stored test_data time values
        time_test = self.test_data['time'].values
        timestamp_test = self.test_data['timestamp'].values

        # Preparar dados
        df_test = self.x_test.copy()
        df_test['time'] = time_test
        df_test['timestamp'] = timestamp_test
        df_test["Real"] = self.y_test.values
        df_test["Predito"] = self.y_pred
        df_test = df_test.sort_values("time")

        df_test["diff"] = df_test["Real"] - df_test["Predito"]
        mean_diff = df_test["diff"].mean()
        stardard_dev = df_test["diff"].std()
        low_lim = mean_diff - 1.5 * stardard_dev
        high_lim = mean_diff + 1.5 * stardard_dev

        # Transformar para formato longo (melhor para px.line)
        df_plot = df_test.melt(id_vars="timestamp", value_vars=["diff"], var_name="Tipo", value_name="Temperature Diff")

        # Grafico Previsões
        fig_pred = px.line(df_test, x="timestamp", y=["Real", "Predito"], title="Regressão XGBoost: CPU Temp: Predict x Real")
        fig_pred.update_traces(line_width=3, marker=dict(size=8))
        fig_pred.show()

        # Gráfico de Erro
        fig = px.line(df_plot, x="timestamp", y="Temperature Diff", color="Tipo", title="Regressão XGBoost: CPU Temp Difference: Predict x Real", markers = True)
        fig = px.line(df_plot, x="timestamp", y="Temperature Diff")
        fig.update_traces(line_color="orange", line_width=3, marker=dict(size=8))
        fig.update_traces(line=dict(width=3))
        fig.add_hline(y=low_lim, line_width=5, line_color="green", line_dash = "dash", opacity = 0.5)
        fig.add_hline(y=high_lim, line_width=5, line_color="red", line_dash = "dash", opacity = 0.5)
        fig.show()

        # Gráfico Detecções
        df_test["anomaly"] = (df_test["diff"] > high_lim) | (df_test["diff"] < low_lim)
        fig_anom = px.line(df_test, x="timestamp", y="anomaly", color="anomaly", title="Regressão XGBoost: CPU Temp Difference: Predict x Real", markers = True)
        fig_anom.show()

    def evaluate_metrics(self):
        rmse = np.sqrt(mean_squared_error(self.y_test, self.y_pred))
        mape = np.mean(np.abs((self.y_test - self.y_pred) / self.y_test)) * 100
        r2 = r2_score(self.y_test, self.y_pred)
        mae = mean_absolute_error(self.y_test, self.y_pred) 
        
        print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
        print(f"Mean Absolute Percentual Error (MAPE): {mape:.2f}%")
        print(f"R2: {r2:.2f}")
        print(f"Mean Absolute Error (MAE): {mae:.2f}")

    def plot_feature_importance(self, top_n=20):
        if hasattr(self.model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': self.x_test.columns,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False).head(top_n)
            
            fig = px.bar(importance_df, x='importance', y='feature', 
                        orientation='h', title='Top Feature Importances')
            fig.show()

In [ ]:
# EXAMPLE 1: Basic workflow (without time features)
core_predictor = CoreTempRegressor(use_time_features = True)
core_predictor.extract_CPU_data(iterations = 8000, csv = True)
core_predictor.plot_CPU_data()

## Example Usage with Time-Based Features

The new workflow includes feature engineering for better predictions:

In [ ]:
# Training Linear model
core_predictor.fit_predict(model="linear", train_size=0.6)
core_predictor.plot_predictions()
core_predictor.evaluate_metrics()

In [ ]:
# Training XGBoost model
core_predictor.fit_predict(model="xgb", train_size=0.6)
core_predictor.plot_predictions()
core_predictor.evaluate_metrics()

In [ ]:
# Training LightGBM model
core_predictor.fit_predict(model="lightgbm", train_size=0.6)
core_predictor.plot_predictions()
core_predictor.evaluate_metrics()